# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

if not hasattr(tf.keras.metrics.Metric, "reset_states"):
    tf.keras.metrics.Metric.reset_states = tf.keras.metrics.Metric.reset_state

device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
from sklearn.model_selection import train_test_split

def load_mnist_dataset():
    """
    Fetch the MNIST dataset from TensorFlow and perform preprocessing to prepare
    it for CNN classifiers.

    Total samples: 70,000. Default split: 50k train / 10k val / 10k test.
    Images: 28×28 grayscale, normalized and reshaped to (N, 28, 28, 1).
    """
    (x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

    X = np.concatenate([x_train_full, x_test], axis=0).astype(np.float32)
    y = np.concatenate([y_train_full, y_test], axis=0).astype(np.int32)
    X = np.expand_dims(X, axis=-1)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.2857, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # Нормализация
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)

    X_train = (X_train - mean_pixel) / (std_pixel + 1e-8)
    X_val   = (X_val - mean_pixel) / (std_pixel + 1e-8)
    X_test  = (X_test - mean_pixel) / (std_pixel + 1e-8)

    return X_train, y_train, X_val, y_val, X_test, y_test



# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist_dataset()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (50001, 28, 28, 1)
Train labels shape:  (50001,) int32
Validation data shape:  (9999, 28, 28, 1)
Validation labels shape:  (9999,)
Test data shape:  (10000, 28, 28, 1)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[idxs[i:i+B]], self.y[idxs[i:i+B]]) for i in range(0, N, B))

train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 28, 28, 1) (64,)
1 (64, 28, 28, 1) (64,)
2 (64, 28, 28, 1) (64,)
3 (64, 28, 28, 1) (64,)
4 (64, 28, 28, 1) (64,)
5 (64, 28, 28, 1) (64,)
6 (64, 28, 28, 1) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            kernel_initializer=initializer
        )

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer
        )

        self.fc = tf.keras.layers.Dense(
            num_classes,
            kernel_initializer=initializer
        )

        self.relu = tf.keras.layers.Activation('relu')
        self.softmax = tf.keras.layers.Activation('softmax')
        self.flatten = tf.keras.layers.Flatten()

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.flatten(x)
        x = self.fc(x)
        scores = self.softmax(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False, print_every=100):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):

        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_states()
            train_accuracy.reset_states()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_states()
                        val_accuracy.reset_states()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.989293098449707, Accuracy: 4.6875, Val Loss: 2.526822328567505, Val Accuracy: 30.723072052001953
Iteration 100, Epoch 1, Loss: 0.7095218300819397, Accuracy: 77.98577117919922, Val Loss: 0.39839673042297363, Val Accuracy: 87.9087905883789
Iteration 200, Epoch 1, Loss: 0.538400411605835, Accuracy: 83.51213073730469, Val Loss: 0.32923442125320435, Val Accuracy: 90.14901733398438
Iteration 300, Epoch 1, Loss: 0.4578630328178406, Accuracy: 86.03612518310547, Val Loss: 0.3202116787433624, Val Accuracy: 90.35903930664062
Iteration 400, Epoch 1, Loss: 0.4175480604171753, Accuracy: 87.39089965820312, Val Loss: 0.2904929220676422, Val Accuracy: 91.26912689208984
Iteration 500, Epoch 1, Loss: 0.38682812452316284, Accuracy: 88.2952880859375, Val Loss: 0.2560015022754669, Val Accuracy: 92.43924713134766
Iteration 600, Epoch 1, Loss: 0.3648409843444824, Accuracy: 89.03390502929688, Val Loss: 0.26308363676071167, Val Accuracy: 92.36923217773438
Iteration 700, Epoch 1, Lo

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.0634896755218506, Accuracy: 6.25, Val Loss: 2.822571277618408, Val Accuracy: 22.642263412475586
Iteration 100, Epoch 1, Loss: 0.4577445983886719, Accuracy: 86.5253677368164, Val Loss: 0.2008378654718399, Val Accuracy: 94.0093994140625
Iteration 200, Epoch 1, Loss: 0.31290149688720703, Accuracy: 90.7182846069336, Val Loss: 0.1717582792043686, Val Accuracy: 94.79948425292969
Iteration 300, Epoch 1, Loss: 0.2533164322376251, Accuracy: 92.48857879638672, Val Loss: 0.1256377398967743, Val Accuracy: 96.26962280273438
Iteration 400, Epoch 1, Loss: 0.21777139604091644, Accuracy: 93.53179168701172, Val Loss: 0.1088675856590271, Val Accuracy: 96.54965209960938
Iteration 500, Epoch 1, Loss: 0.19556277990341187, Accuracy: 94.18350982666016, Val Loss: 0.10422088205814362, Val Accuracy: 97.03970336914062
Iteration 600, Epoch 1, Loss: 0.17806407809257507, Accuracy: 94.67813873291016, Val Loss: 0.09258580952882767, Val Accuracy: 97.40974426269531
Iteration 700, Epoch 1, L

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (28, 28, 1)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.9469542503356934, Accuracy: 15.625, Val Loss: 2.648658275604248, Val Accuracy: 19.681968688964844
Iteration 100, Epoch 1, Loss: 0.6976948976516724, Accuracy: 77.98577117919922, Val Loss: 0.42642804980278015, Val Accuracy: 87.25872802734375
Iteration 200, Epoch 1, Loss: 0.5330430269241333, Accuracy: 83.5198974609375, Val Loss: 0.3341832160949707, Val Accuracy: 90.18901824951172
Iteration 300, Epoch 1, Loss: 0.4584723711013794, Accuracy: 86.0465087890625, Val Loss: 0.30145159363746643, Val Accuracy: 90.88909149169922
Iteration 400, Epoch 1, Loss: 0.4152606129646301, Accuracy: 87.46882629394531, Val Loss: 0.30054429173469543, Val Accuracy: 91.2491226196289
Iteration 500, Epoch 1, Loss: 0.384290486574173, Accuracy: 88.50424194335938, Val Loss: 0.266012579202652, Val Accuracy: 92.10920715332031
Iteration 600, Epoch 1, Loss: 0.36262384057044983, Accuracy: 89.20289611816406, Val Loss: 0.25179263949394226, Val Accuracy: 92.60926055908203
Iteration 700, Epoch 1, Lo

Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.3340 - sparse_categorical_accuracy: 0.9016 - val_loss: 0.4594 - val_sparse_categorical_accuracy: 0.8400
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4546 - sparse_categorical_accuracy: 0.8460


[0.4546297788619995, 0.8460000157356262]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(channel_1, kernel_size=5, padding='same', kernel_initializer=initializer),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Conv2D(channel_2, kernel_size=3, padding='same', kernel_initializer=initializer),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(num_classes, kernel_initializer=initializer),
        tf.keras.layers.Activation('softmax')
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.7667593955993652, Accuracy: 9.375, Val Loss: 2.605475664138794, Val Accuracy: 11.2811279296875
Iteration 100, Epoch 1, Loss: 0.7048613429069519, Accuracy: 77.19678497314453, Val Loss: 0.338289350271225, Val Accuracy: 89.77897644042969
Iteration 200, Epoch 1, Loss: 0.5044660568237305, Accuracy: 84.032958984375, Val Loss: 0.27873486280441284, Val Accuracy: 91.6191635131836
Iteration 300, Epoch 1, Loss: 0.42080673575401306, Accuracy: 86.89783477783203, Val Loss: 0.23899179697036743, Val Accuracy: 92.84928131103516
Iteration 400, Epoch 1, Loss: 0.36875271797180176, Accuracy: 88.61050415039062, Val Loss: 0.20626257359981537, Val Accuracy: 93.78938293457031
Iteration 500, Epoch 1, Loss: 0.3328547179698944, Accuracy: 89.78916931152344, Val Loss: 0.20106413960456848, Val Accuracy: 93.98939514160156
Iteration 600, Epoch 1, Loss: 0.30738213658332825, Accuracy: 90.54960632324219, Val Loss: 0.18775279819965363, Val Accuracy: 94.17941284179688
Iteration 700, Epoch 1, L

In [14]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.2296 - sparse_categorical_accuracy: 0.9341 - val_loss: 0.1221 - val_sparse_categorical_accuracy: 0.9624
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1216 - sparse_categorical_accuracy: 0.9655


[0.12158902734518051, 0.965499997138977]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (28, 28, 1)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.966367244720459, Accuracy: 14.0625, Val Loss: 2.660731077194214, Val Accuracy: 16.761674880981445
Iteration 100, Epoch 1, Loss: 0.6634377241134644, Accuracy: 79.6875, Val Loss: 0.4135020971298218, Val Accuracy: 87.84878540039062
Iteration 200, Epoch 1, Loss: 0.5224648714065552, Accuracy: 84.43719482421875, Val Loss: 0.35296860337257385, Val Accuracy: 89.31893157958984
Iteration 300, Epoch 1, Loss: 0.4516221284866333, Accuracy: 86.57080841064453, Val Loss: 0.31555312871932983, Val Accuracy: 90.63906860351562
Iteration 400, Epoch 1, Loss: 0.4072243869304657, Accuracy: 88.06499481201172, Val Loss: 0.2769019305706024, Val Accuracy: 91.7791748046875
Iteration 500, Epoch 1, Loss: 0.37679895758628845, Accuracy: 88.99076843261719, Val Loss: 0.2644515335559845, Val Accuracy: 92.09920501708984
Iteration 600, Epoch 1, Loss: 0.35622015595436096, Accuracy: 89.54606628417969, Val Loss: 0.24949905276298523, Val Accuracy: 92.48925018310547
Iteration 700, Epoch 1, Loss: 0.

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

**Эксперемент 1**

Двухблочная свёрточная сеть (32->64 фильтра) с пакетной нормализацией и пулингом максимумов. Агрегация признаков через `GlobalAveragePooling2D` выполняет структурную регуляризацию, исключая параметрический `Flatten` и сокращая число обучаемых параметров. Оптимизация алгоритмом `Adam` (lr=1e-3).

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        init = tf.keras.initializers.HeNormal()

        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same', kernel_initializer=init)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', kernel_initializer=init)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc = tf.keras.layers.Dense(10, kernel_initializer=init)

    def call(self, x, training=False):
        x = tf.nn.relu(self.bn1(self.conv1(x), training=training))
        x = self.pool(x)
        x = tf.nn.relu(self.bn2(self.conv2(x), training=training))
        x = self.pool(x)
        x = self.gap(x)
        return self.fc(x)

model1 = CustomConvNet()
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

history1 = model1.fit(
    X_train, y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1
)

test_loss, test_acc = model1.evaluate(X_test, y_test, verbose=0)
print(f"Model 1 - Test Accuracy: {test_acc:.4f}")

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 1.4337 - sparse_categorical_accuracy: 0.6463 - val_loss: 1.2349 - val_sparse_categorical_accuracy: 0.5787
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.7418 - sparse_categorical_accuracy: 0.8609 - val_loss: 0.7714 - val_sparse_categorical_accuracy: 0.7588
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4756 - sparse_categorical_accuracy: 0.9070 - val_loss: 0.4936 - val_sparse_categorical_accuracy: 0.8752
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3540 - sparse_categorical_accuracy: 0.9261 - val_loss: 0.3940 - val_sparse_categorical_accuracy: 0.8953
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2862 - sparse_categorical_accuracy: 0.9379 - val_loss: 0.3003 - val_sparse_categorical_accuracy: 0.9300
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.2426 - sparse_categorical_accuracy: 0.9453 - val_loss: 0.3057 - val_sparse_categorical_accuracy: 0.9181
Ep

**Эксперемент 2**

Трёхслойная иерархия (16->32->64 фильтра) с явной регуляризацией: L2-штраф (1e-4) и Dropout (0.15). Оптимизация SGD с импульсом Нестерова.

In [18]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        init = tf.keras.initializers.HeNormal()
        reg = tf.keras.regularizers.l2(1e-4)

        self.conv1 = tf.keras.layers.Conv2D(16, 3, padding='same', kernel_initializer=init, kernel_regularizer=reg)
        self.conv2 = tf.keras.layers.Conv2D(32, 3, padding='same', kernel_initializer=init, kernel_regularizer=reg)
        self.conv3 = tf.keras.layers.Conv2D(64, 3, padding='same', kernel_initializer=init, kernel_regularizer=reg)
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.flatten = tf.keras.layers.Flatten()
        self.dropout = tf.keras.layers.Dropout(0.15)
        self.fc = tf.keras.layers.Dense(10, kernel_initializer=init, kernel_regularizer=reg)

    def call(self, x, training=False):
        x = tf.nn.relu(self.conv1(x))
        x = tf.nn.relu(self.conv2(self.pool(x)))
        x = tf.nn.relu(self.pool(self.conv3(x)))
        x = self.flatten(x)
        x = self.dropout(x, training=training)
        return self.fc(x)

model2 = CustomConvNet()
model2.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9, nesterov=True),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

history2 = model2.fit(
    X_train, y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1
)

test_loss, test_acc = model2.evaluate(X_test, y_test, verbose=0)
print(f"Model 2 - Test Accuracy: {test_acc:.4f}")

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 0.3686 - sparse_categorical_accuracy: 0.8956 - val_loss: 0.1573 - val_sparse_categorical_accuracy: 0.9621
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1506 - sparse_categorical_accuracy: 0.9613 - val_loss: 0.1169 - val_sparse_categorical_accuracy: 0.9720
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1196 - sparse_categorical_accuracy: 0.9707 - val_loss: 0.1010 - val_sparse_categorical_accuracy: 0.9770
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1030 - sparse_categorical_accuracy: 0.9760 - val_loss: 0.0922 - val_sparse_categorical_accuracy: 0.9799
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0938 - sparse_categorical_accuracy: 0.9780 - val_loss: 0.0871 - val_sparse_categorical_accuracy: 0.9813
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0861 - sparse_categorical_accuracy: 0.9807 - val_loss: 0.0833 - val_sparse_categorical_accuracy: 0.9816
Epo

Опишите все эксперименты, результаты. Сделайте выводы.

**Эксперимент 1**

Модель демонстрирует быструю начальную сходимость с последующей стабилизацией: точность на валидации достигает локального максимума 94,53% на 8-й эпохе, после чего незначительно снижается, завершая обучение на уровне 93,21%. Финальная точность на тестовой выборке составляет 93,25%. К 6-7-й эпохе наблюдается расхождение кривых функции потерь (обучающая монотонно снижается до 0,156, тогда как валидационная стабилизируется около 0,239), что указывает на начало умеренного переобучения. Операция `GlobalAveragePooling2D`, эффективно снижающая число параметров на сложных датасетах, в условиях MNIST приводит к избыточному сжатию пространственных признаков, ограничивая предельную ёмкость классификатора и препятствуя дальнейшему росту точности после 5-й эпохи.

**Эксперимент 2**

Архитектура показывает стабильную и монотонную динамику на протяжении всех 10 эпох: точность на валидации уверенно возрастает с 96,21% до 98,40%, а итоговая метрика на тесте достигает 98,41%. Кривые обучающей и валидационной функции потерь снижаются синхронно, расхождение между ними к концу обучения составляет менее 0,5%, что свидетельствует об эффективном контроле обобщающей способности. Комбинация L2-регуляризации, умеренного Dropout и оптимизатора SGD с импульсом Нестерова обеспечивает плавную настройку весов, предотвращая как переобучение, так и преждевременную стагнацию, что позволяет модели полностью раскрыть свой потенциал.

**Основной вывод**

Сравнительный анализ подтверждает, что для задач с низкой вариативностью данных, таких как MNIST, предпочтение следует отдавать архитектурам с явной регуляризацией и детерминированными оптимизаторами.